In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
import matplotlib.pyplot as plt

# Add project root to sys.path to import local modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_loader import MPDDataLoader
from embedding_model import EmbeddingGenerator
from src.cf_model import CFRecommender
from src.content_model import ContentRecommender
from src.hybrid_model import HybridRecommender
from src.reranker import FairnessReranker

# Load MPD playlists (adjust max_files for testing)
loader = MPDDataLoader(data_folder='../data/')
playlists = loader.load_playlists(max_files=50)
track_df = loader.build_track_df(playlists)

# Prepare track_id column for merging
track_df['track_id'] = track_df['track_uri'].apply(lambda x: x.split(':')[-1])

print("Loaded tracks:", track_df.shape)
track_df.head()


Loaded tracks: (3302642, 8)


,pid,track_uri,track_name,artist_uri,artist_name,album_uri,album_name,track_id
0,115000,spotify:track:0ESJlaM8CE1jRWaNtwSNj8,beibs in the trap,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:42WVQWuf1teDysXiOupIZt,Birds In The Trap Sing McKnight,0ESJlaM8CE1jRWaNtwSNj8
1,115000,spotify:track:6gBFPUFcJLzWGx4lenP6h2,goosebumps,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:42WVQWuf1teDysXiOupIZt,Birds In The Trap Sing McKnight,6gBFPUFcJLzWGx4lenP6h2
2,115000,spotify:track:2c2csx4OTYtbkzvbSTXlGY,guidance,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:42WVQWuf1teDysXiOupIZt,Birds In The Trap Sing McKnight,2c2csx4OTYtbkzvbSTXlGY
3,115000,spotify:track:1yxgsra98r3qAtxqiGZPiX,Butterfly Effect,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:4fOw7xSDwqb58Z2Qia5j81,Butterfly Effect,1yxgsra98r3qAtxqiGZPiX
4,115000,spotify:track:1SGt65i9AnXYdDQt1AtDRH,3500,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:4PWBTB6NYSKQwfo79I3prg,Rodeo,1SGt65i9AnXYdDQt1AtDRH


In [24]:
# Skip actual Spotify API calls due to 403 errors
# Instead create empty audio_features_df with expected columns

audio_features_df = pd.DataFrame(columns=[
    'id', 'danceability', 'energy', 'valence', 'tempo'
])

# Add track_id column to audio_features_df to merge
audio_features_df['track_id'] = []

# Merge with track_df (will have NaNs for audio features)
merged_df = pd.merge(track_df, audio_features_df, on='track_id', how='left')

print("Merged DF shape:", merged_df.shape)
merged_df.head()


Merged DF shape: (3302642, 13)


,pid,track_uri,track_name,artist_uri,artist_name,album_uri,album_name,track_id,id,danceability,energy,valence,tempo
0,115000,spotify:track:0ESJlaM8CE1jRWaNtwSNj8,beibs in the trap,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:42WVQWuf1teDysXiOupIZt,Birds In The Trap Sing McKnight,0ESJlaM8CE1jRWaNtwSNj8,NaN,NaN,NaN,NaN,NaN
1,115000,spotify:track:6gBFPUFcJLzWGx4lenP6h2,goosebumps,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:42WVQWuf1teDysXiOupIZt,Birds In The Trap Sing McKnight,6gBFPUFcJLzWGx4lenP6h2,NaN,NaN,NaN,NaN,NaN
2,115000,spotify:track:2c2csx4OTYtbkzvbSTXlGY,guidance,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:42WVQWuf1teDysXiOupIZt,Birds In The Trap Sing McKnight,2c2csx4OTYtbkzvbSTXlGY,NaN,NaN,NaN,NaN,NaN
3,115000,spotify:track:1yxgsra98r3qAtxqiGZPiX,Butterfly Effect,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:4fOw7xSDwqb58Z2Qia5j81,Butterfly Effect,1yxgsra98r3qAtxqiGZPiX,NaN,NaN,NaN,NaN,NaN
4,115000,spotify:track:1SGt65i9AnXYdDQt1AtDRH,3500,spotify:artist:0Y5tJX1MQlPlqiwlOH1tJY,Travis Scott,spotify:album:4PWBTB6NYSKQwfo79I3prg,Rodeo,1SGt65i9AnXYdDQt1AtDRH,NaN,NaN,NaN,NaN,NaN


In [6]:
# # %%
# # Setup Spotify API client (put your credentials here)
# CLIENT_ID = "8d0b81b7efd94d28be4b8e5cfd9bbad5"
# CLIENT_SECRET = "a26d760401044cb6bf19a0c03e444b89"
# api = SpotifyAPI(CLIENT_ID, CLIENT_SECRET)

# # Test call - get audio features for first 10 unique tracks (handle 403 errors gracefully)
# track_uris = track_df['track_uri'].unique().tolist()
# try:
#     features_list = api.get_multiple_audio_features(track_uris[:10])
#     audio_features_df = pd.DataFrame(features_list)
#     print("Audio features fetched from Spotify API.")
# except Exception as e:
#     print(f"Warning: Spotify API audio feature fetch failed: {e}")
#     # Create empty dataframe with expected columns as fallback
#     audio_features_df = pd.DataFrame(columns=['id', 'danceability', 'energy', 'valence', 'tempo', 'track_id'])


In [25]:
# Fill missing audio feature values with zeros before scaling
feature_cols = ['danceability', 'energy', 'valence', 'tempo']
merged_df[feature_cols] = merged_df[feature_cols].fillna(0)

# Scale features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(merged_df[feature_cols])

# Replace any NaNs in scaled features (safety)
features_scaled = np.nan_to_num(features_scaled)

# Add scaled features back to merged_df for aggregation
for i, col in enumerate(feature_cols):
    merged_df[col + '_scaled'] = features_scaled[:, i]

# Playlist embeddings: average scaled features per playlist
playlist_embeddings_df = merged_df.groupby('pid')[[col + '_scaled' for col in feature_cols]].mean()

playlist_embeddings = playlist_embeddings_df.values
playlist_embeddings = np.nan_to_num(playlist_embeddings)

# Track embeddings from scaled features
track_embeddings = features_scaled

# Track URIs list aligned with track_embeddings rows
track_uris = merged_df['track_uri'].tolist()

print(f"Playlist embeddings shape: {playlist_embeddings.shape}")
print(f"Track embeddings shape: {track_embeddings.shape}")


/var/folders/x3/xlf84vmx7xv6jybnkt035hzh0000gn/T/ipykernel_71806/2430694963.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged_df[feature_cols] = merged_df[feature_cols].fillna(0)


Playlist embeddings shape: (50000, 4)
Track embeddings shape: (3302642, 4)


In [26]:
embedder = EmbeddingGenerator()

# Calculate popularity mapping (track_uri -> popularity)
popularity = embedder.track_popularity(track_df)

# For playlists, create dummy names and title embeddings (optional)
playlists_df = track_df.groupby('pid').first().reset_index()
playlists_df['name'] = 'my playlist'

playlists = playlists_df.to_dict(orient='records')
playlist_title_embeddings = embedder.playlist_title_embeddings(playlists)

print(f"Playlist title embeddings shape: {playlist_title_embeddings.shape}")


Playlist title embeddings shape: (50000, 2)


In [27]:
cf = CFRecommender()
cf.train(track_df)

print(f"user_item_matrix shape: {cf.user_item_matrix.shape}")


  0%|          | 0/10 [00:00<?, ?it/s]

user_item_matrix shape: (50000, 462106)


In [29]:
playlist_indices = cf.playlist_codes.unique()[:5]

for playlist_idx in playlist_indices:
    print(f"Playlist idx: {playlist_idx}")
    
    pid = cf.playlist_categories[playlist_idx]
    print(f"Playlist category ID: {pid}")
    
    ground_truth_tracks = track_df[track_df['pid'] == pid]['track_uri'].unique()
    print(f"Number of ground truth tracks: {len(ground_truth_tracks)}")
    print(f"Sample ground truth tracks: {ground_truth_tracks[:5]}")
    
    cf_recs = cf.recommend_for_playlist(playlist_idx, N=10)
    rec_uris = [t[0] for t in cf_recs]  # FIXED here
    
    print(f"Number of CF recommended tracks: {len(rec_uris)}")
    print(f"Sample recommended tracks: {rec_uris[:5]}")
    
    common = set(rec_uris) & set(ground_truth_tracks)
    print(f"Common tracks between CF recs and ground truth: {len(common)}\n")


Playlist idx: 5000
Playlist category ID: 115000
Number of ground truth tracks: 211
Sample ground truth tracks: ['spotify:track:0ESJlaM8CE1jRWaNtwSNj8'
 'spotify:track:6gBFPUFcJLzWGx4lenP6h2'
 'spotify:track:2c2csx4OTYtbkzvbSTXlGY'
 'spotify:track:1yxgsra98r3qAtxqiGZPiX'
 'spotify:track:1SGt65i9AnXYdDQt1AtDRH']
Number of CF recommended tracks: 10
Sample recommended tracks: ['spotify:track:0SGkqnVQo9KPytSri1H6cF', 'spotify:track:1f5cbQtDrykjarZVrShaDI', 'spotify:track:7wwifjNAb172PtDpKK3CoR', 'spotify:track:31Q9ZTF9x81BDonlObCbvP', 'spotify:track:7hDc8b7IXETo14hHIHdnhd']
Common tracks between CF recs and ground truth: 0

Playlist idx: 5001
Playlist category ID: 115001
Number of ground truth tracks: 165
Sample ground truth tracks: ['spotify:track:1wRCg1JyRmbJEg9ESOMlVp'
 'spotify:track:6GsmDh7AXio3QbYM8m4pul'
 'spotify:track:6FqNRGxwUXn6n3YKZJ06hX'
 'spotify:track:329ajST50FgS9VdSpoofvg'
 'spotify:track:2y42MmBCfEhqIkFAJs0V8Z']
Number of CF recommended tracks: 10
Sample recommended tracks

In [39]:
# Retrain CF model with tuned hyperparameters
cf = CFRecommender(factors=50, regularization=0.1, iterations=30)
cf.train(track_df)

# Debugging CF model mappings and data

print(f"User-item matrix shape: {cf.user_item_matrix.shape}")
print(f"Number of playlists in model: {len(cf.playlist_codes)}")
print(f"Sample playlist codes: {cf.playlist_codes[:5]}")
print(f"Playlist mapping keys sample: {list(cf.playlist_mapping.keys())[:5]}")
print(f"Playlist mapping values sample: {list(cf.playlist_mapping.values())[:5]}")

# Check example playlist row sum (sparsity check)
example_idx = 0
print(f"Example playlist matrix row sum (index {example_idx}): {cf.user_item_matrix[example_idx].sum()}")


# Test recommendations after retraining
playlist_idx = 0  # Example playlist
cf_scores = dict(cf.recommend_for_playlist(playlist_idx, N=20))

print("Top CF recommendations after tuning:")
for track_uri, score in list(cf_scores.items())[:5]:
    print(f"{track_uri}: {score:.3f}")


  0%|          | 0/30 [00:00<?, ?it/s]

User-item matrix shape: (50000, 462106)
Number of playlists in model: 3302642
Sample playlist codes: 0    5000
1    5000
2    5000
3    5000
4    5000
dtype: int32
Playlist mapping keys sample: [0, 1, 2, 3, 4]
Playlist mapping values sample: [3000, 3001, 3002, 3003, 3004]
Example playlist matrix row sum (index 0): 16.0
Top CF recommendations after tuning:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.002
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.002
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.002
spotify:track:4WjH9Bzt3kx7z8kl0awxh4: 0.002
spotify:track:6YUTL4dYpB9xZO5qExPf05: 0.002


In [31]:
content = ContentRecommender(playlist_embeddings, track_embeddings, track_uris)


In [32]:
hybrid = HybridRecommender(cf_model=cf, content_model=content, alpha=0.7)


In [33]:
# Tune alpha hyperparameter for Hybrid model

alphas = [0.1, 0.3, 0.5, 0.7, 0.9]
playlist_idx = 0  # Example playlist

for alpha in alphas:
    hybrid = HybridRecommender(cf_model=cf, content_model=content, alpha=alpha)
    hybrid_scores = dict(hybrid.recommend_for_playlist(playlist_idx, N=20))
    
    print(f"Alpha={alpha} - Top 3 hybrid recommendations:")
    top_scores = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)[:3]
    for uri, score in top_scores:
        print(f"{uri}: {score:.4f}")
    print()


Alpha=0.1 - Top 3 hybrid recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0002
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0002
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0002

Alpha=0.3 - Top 3 hybrid recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0007
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0006
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0006

Alpha=0.5 - Top 3 hybrid recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0012
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0010
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0010

Alpha=0.7 - Top 3 hybrid recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0016
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0014
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0014

Alpha=0.9 - Top 3 hybrid recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0021
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0019
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0018



In [34]:
reranker = FairnessReranker(popularity, beta=0.5, epsilon=1e-6)


In [35]:
# Tune beta hyperparameter for reranker

betas = [0.1, 0.3, 0.5, 0.7, 0.9]
playlist_idx = 0  # Example playlist

for beta in betas:
    reranker = FairnessReranker(popularity, beta=beta, epsilon=1e-6)
    hybrid_recs = hybrid.recommend_for_playlist(playlist_idx, N=50)
    reranked_recs = reranker.rerank(hybrid_recs)
    
    print(f"Beta={beta} - Top 3 reranked recommendations:")
    for uri, score in reranked_recs[:3]:
        print(f"{uri}: {score:.4f}")
    print()


Beta=0.1 - Top 3 reranked recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0017
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0015
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0015

Beta=0.3 - Top 3 reranked recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0011
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0011
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0010

Beta=0.5 - Top 3 reranked recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0008
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0007
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0007

Beta=0.7 - Top 3 reranked recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0005
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0005
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0005

Beta=0.9 - Top 3 reranked recommendations:
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0003
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0003
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0003



In [40]:
playlist_idx = 0  # change to test different playlists
pid = cf.playlist_categories[playlist_idx]

print(f"Playlist idx: {playlist_idx}")
print(f"Playlist category ID: {pid}")

ground_truth_tracks = track_df[track_df['pid'] == pid]['track_uri'].unique()
print(f"Number of ground truth tracks: {len(ground_truth_tracks)}")
print(f"Sample ground truth tracks: {ground_truth_tracks[:5]}")

cf_recs = cf.recommend_for_playlist(playlist_idx, N=50)  # increased N to 50
rec_uris = [t[0] for t in cf_recs]

print(f"Number of CF recommended tracks: {len(rec_uris)}")
print(f"Sample recommended tracks: {rec_uris[:5]}")

common_tracks = set(rec_uris) & set(ground_truth_tracks)
print(f"Common tracks between CF recs and ground truth: {len(common_tracks)}")
print(f"Sample common tracks: {list(common_tracks)[:5]}")


Playlist idx: 0
Playlist category ID: 3000
Number of ground truth tracks: 16
Sample ground truth tracks: ['spotify:track:2vCtiBvJJZfz773yTfAxPP'
 'spotify:track:5WOLZP8KrXiupBjG1SSN5U'
 'spotify:track:0oQDQ9QiqsO63EEBAro8Le'
 'spotify:track:7MUS0La2IQ85vJ59fQqtoN'
 'spotify:track:3ciyZYofjiqmMUElM5qgGB']
Number of CF recommended tracks: 50
Sample recommended tracks: ['spotify:track:7BKLCZ1jbUBVqRi2FVlTVw', 'spotify:track:04DwTuZ2VBdJCCC5TROn7L', 'spotify:track:46lFttIf5hnUZMGvjK0Wxo', 'spotify:track:4WjH9Bzt3kx7z8kl0awxh4', 'spotify:track:6YUTL4dYpB9xZO5qExPf05']
Common tracks between CF recs and ground truth: 0
Sample common tracks: []


In [36]:
playlist_idx = 0  # Example playlist index

print("CF Recommendations:")
cf_scores = dict(cf.recommend_for_playlist(playlist_idx, N=20))
for track_uri, score in list(cf_scores.items())[:5]:
    print(f"{track_uri}: {score:.3f}")

print("\nContent Recommendations:")
content_scores = dict(content.recommend_for_playlist(playlist_idx, N=20))
for track_uri, score in list(content_scores.items())[:5]:
    print(f"{track_uri}: {score:.3f}")

print("\nHybrid Recommendations:")
hybrid_scores = dict(hybrid.recommend_for_playlist(playlist_idx, N=20))
for track_uri, score in list(hybrid_scores.items())[:5]:
    print(f"{track_uri}: {score:.3f}")

print("\nReranked Recommendations:")
reranked_recs = reranker.rerank(list(hybrid_scores.items()))
for track_uri, score in reranked_recs[:5]:
    print(f"{track_uri}: {score:.4f}")


CF Recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.002
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.002
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.002
spotify:track:4WjH9Bzt3kx7z8kl0awxh4: 0.002
spotify:track:6YUTL4dYpB9xZO5qExPf05: 0.002

Content Recommendations:
spotify:track:6H0AwSQ20mo62jGlPGB8S6: 0.000
spotify:track:2p3ByN7qyhZD2nM8zSVsil: 0.000
spotify:track:5dUxIdq8o7HSBDYqIEXcNR: 0.000
spotify:track:5OM1p9cVPxzUxN0HfRYvWh: 0.000
spotify:track:7MSuEyj8woELvqNYNc6OtR: 0.000

Hybrid Recommendations:
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.002
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.002
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.002
spotify:track:4WjH9Bzt3kx7z8kl0awxh4: 0.002
spotify:track:6YUTL4dYpB9xZO5qExPf05: 0.002

Reranked Recommendations:
spotify:track:46lFttIf5hnUZMGvjK0Wxo: 0.0003
spotify:track:7BKLCZ1jbUBVqRi2FVlTVw: 0.0003
spotify:track:04DwTuZ2VBdJCCC5TROn7L: 0.0003
spotify:track:6YUTL4dYpB9xZO5qExPf05: 0.0003
spotify:track:4WjH9Bzt3kx7z8kl0awxh4: 0.0003


In [37]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if relevant else 0

def ndcg_at_k(recommended, relevant, k):
    relevance = [1 if item in relevant else 0 for item in recommended[:k]]
    ideal_relevance = sorted(relevance, reverse=True)
    return ndcg_score([ideal_relevance], [relevance])


In [38]:
N = 20
K = 20
playlist_indices = cf.playlist_codes.unique()[:10]  # evaluate on first 10 playlists

cf_precisions, content_precisions, hybrid_precisions, reranked_precisions = [], [], [], []
cf_recalls, content_recalls, hybrid_recalls, reranked_recalls = [], [], [], []
cf_ndcgs, content_ndcgs, hybrid_ndcgs, reranked_ndcgs = [], [], [], []

for playlist_idx in playlist_indices:
    cf_recs = list(cf.recommend_for_playlist(playlist_idx, N))
    content_recs = list(content.recommend_for_playlist(playlist_idx, N))
    hybrid_recs = list(hybrid.recommend_for_playlist(playlist_idx, N))
    reranked_recs = [t[0] for t in reranker.rerank(hybrid_recs)[:K]]

    cf_uris = [t[0] for t in cf_recs]
    content_uris = [t[0] for t in content_recs]
    hybrid_uris = [t[0] for t in hybrid_recs]

    ground_truth = set(track_df[track_df['pid'] == cf.playlist_categories[playlist_idx]]['track_uri'])
    
    cf_precisions.append(precision_at_k(cf_uris, ground_truth, K))
    content_precisions.append(precision_at_k(content_uris, ground_truth, K))
    hybrid_precisions.append(precision_at_k(hybrid_uris, ground_truth, K))
    reranked_precisions.append(precision_at_k(reranked_recs, ground_truth, K))

    cf_recalls.append(recall_at_k(cf_uris, ground_truth, K))
    content_recalls.append(recall_at_k(content_uris, ground_truth, K))
    hybrid_recalls.append(recall_at_k(hybrid_uris, ground_truth, K))
    reranked_recalls.append(recall_at_k(reranked_recs, ground_truth, K))

    cf_ndcgs.append(ndcg_at_k(cf_uris, ground_truth, K))
    content_ndcgs.append(ndcg_at_k(content_uris, ground_truth, K))
    hybrid_ndcgs.append(ndcg_at_k(hybrid_uris, ground_truth, K))
    reranked_ndcgs.append(ndcg_at_k(reranked_recs, ground_truth, K))

print(f"CF Precision@{K}: {np.mean(cf_precisions):.3f}, Recall@{K}: {np.mean(cf_recalls):.3f}, NDCG@{K}: {np.mean(cf_ndcgs):.3f}")
print(f"Content Precision@{K}: {np.mean(content_precisions):.3f}, Recall@{K}: {np.mean(content_recalls):.3f}, NDCG@{K}: {np.mean(content_ndcgs):.3f}")
print(f"Hybrid Precision@{K}: {np.mean(hybrid_precisions):.3f}, Recall@{K}: {np.mean(hybrid_recalls):.3f}, NDCG@{K}: {np.mean(hybrid_ndcgs):.3f}")
print(f"Reranked Precision@{K}: {np.mean(reranked_precisions):.3f}, Recall@{K}: {np.mean(reranked_recalls):.3f}, NDCG@{K}: {np.mean(reranked_ndcgs):.3f}")


CF Precision@20: 0.000, Recall@20: 0.000, NDCG@20: 0.000
Content Precision@20: 0.010, Recall@20: 0.005, NDCG@20: 0.200
Hybrid Precision@20: 0.000, Recall@20: 0.000, NDCG@20: 0.000
Reranked Precision@20: 0.000, Recall@20: 0.000, NDCG@20: 0.000
